# Evaluating and Optimizing Prompts with DSPy

## Welcome!

You have a dataset of **300 US cities** with population and land area data. You want an LLM to answer factual questions about them — accurately. Instead of hand-writing prompts, you'll **compile** them with DSPy.

### What you'll build
A **city Q&A system** that answers population and population-density questions, scored against ground-truth data.

### What you'll learn
1. **Signatures** — typed input/output contracts for LLMs
2. **Evaluation** — systematic scoring with `dspy.Evaluate`
3. **Optimization** — automatic prompt tuning with `BootstrapFewShot`
4. **ChainOfThought** — adding reasoning steps
5. **Modules** — composable building blocks
6. **LLM-as-Judge** — combining numeric metrics with LLM-based faithfulness checks

## 1. Install Dependencies

DSPy handles LLM orchestration and optimization. We also need `pandas` to load the city data.

In [ ]:
!pip install dspy boto3 pandas --quiet

## 2. Configure DSPy with Amazon Bedrock

DSPy supports Bedrock natively via LiteLLM. We configure a single LM that all modules will use.

In [ ]:
import dspy

lm = dspy.LM("bedrock/global.anthropic.claude-haiku-4-5-20251001-v1:0")
dspy.configure(lm=lm)

# Quick sanity check
response = lm("Say 'hello' and nothing else.")
print(f"Connection OK: {response}")

## 3. Load the City Data

Load `city_pop.csv`, clean up footnote markers in city names, parse population values, and compute population density.

In [ ]:
import pandas as pd
import re

df = pd.read_csv("city_pop.csv")

# Clean city names — strip footnote markers like [c], [d]
df["city"] = df["city"].apply(lambda x: re.sub(r"\[.*?\]", "", x).strip())

# Parse population — remove commas
df["population"] = df["population"].apply(lambda x: int(str(x).replace(",", "")))

# Calculate density (people per square mile)
df["density"] = (df["population"] / df["land_area_mi2"]).round(1)

print(f"Loaded {len(df)} cities")
df[["city", "state", "population", "land_area_mi2", "density"]].head(10)

## 4. Prepare DSPy Examples

Generate question/answer pairs from the dataframe — one population question and one density question per city. Split into training (first 20 cities → 40 examples) and test (next 10 cities → 20 examples).

In [ ]:
def make_examples(df_slice):
    examples = []
    for _, row in df_slice.iterrows():
        examples.append(dspy.Example(
            question=f"What is the population of {row['city']}, {row['state']}?",
            answer=str(row["population"])
        ).with_inputs("question"))
        examples.append(dspy.Example(
            question=f"What is the population density (people per square mile) of {row['city']}, {row['state']}?",
            answer=str(row["density"])
        ).with_inputs("question"))
    return examples

trainset = make_examples(df.iloc[:20])
testset = make_examples(df.iloc[20:30])

print(f"Training examples: {len(trainset)}")
print(f"Test examples: {len(testset)}")
print(f"\nSample: {trainset[0].question} → {trainset[0].answer}")

## 5. Define a DSPy Signature

A **Signature** declares the input/output contract. DSPy auto-generates the prompt from the docstring and field descriptions.

In [ ]:
class CityQA(dspy.Signature):
    """Answer factual questions about US cities using your knowledge."""
    question: str = dspy.InputField(desc="A question about a US city's population or demographics")
    answer: str = dspy.OutputField(desc="A precise numeric answer to the question")

## 6. First Prediction (Baseline)

Run the signature with `dspy.Predict` — no optimization, no few-shot examples. This is our baseline.

In [ ]:
baseline = dspy.Predict(CityQA)

test_example = testset[0]
result = baseline(question=test_example.question)

print(f"Question: {test_example.question}")
print(f"Predicted: {result.answer}")
print(f"Expected:  {test_example.answer}")

### Inspect the Auto-Generated Prompt

Let's see what DSPy actually sent to the LLM.

In [ ]:
dspy.inspect_history(n=1)

## 7. Define a Quality Metric

Extract the numeric value from the model's answer, compute percentage error against ground truth. Scoring: within 5% → 1.0, within 10% → 0.5, else → 0.0. During optimization (`trace is not None`), return a boolean.

In [ ]:
def extract_number(text):
    """Extract the first number from text, handling commas and decimals."""
    text = str(text).replace(",", "")
    matches = re.findall(r"[\d]+\.?[\d]*", text)
    return float(matches[0]) if matches else None

def city_metric(example, prediction, trace=None):
    """Score based on % error: within 5% → 1.0, within 10% → 0.5, else → 0.0."""
    expected = extract_number(example.answer)
    predicted = extract_number(prediction.answer)

    if expected is None or predicted is None or expected == 0:
        return False if trace else 0.0

    pct_error = abs(predicted - expected) / expected

    if pct_error <= 0.05:
        score = 1.0
    elif pct_error <= 0.10:
        score = 0.5
    else:
        score = 0.0

    if trace is not None:
        return score >= 0.5
    return score

# Quick test
score = city_metric(test_example, result)
print(f"Score: {score}")

## 8. Baseline Evaluation

Evaluate the baseline predictor across all test examples.

In [ ]:
evaluator = dspy.Evaluate(
    devset=testset,
    metric=city_metric,
    num_threads=2,
    display_progress=True,
    display_table=5,
)

baseline_score = evaluator(baseline)
print(f"\nBaseline average score: {baseline_score}")

## 9. Optimize with BootstrapFewShot

`BootstrapFewShot` finds effective few-shot examples from the training set. It runs the model on training examples, keeps the ones that pass the metric, and injects them as demonstrations.

In [ ]:
optimizer = dspy.BootstrapFewShot(
    metric=city_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)

optimized = optimizer.compile(baseline, trainset=trainset)

optimized_score = evaluator(optimized)

print(f"\n{'='*40}")
print(f"Baseline score:  {baseline_score}")
print(f"Optimized score: {optimized_score}")
print(f"{'='*40}")

## 10. Inspect the Optimized Program

What did the optimizer actually produce? Let's look at the few-shot demos it selected and the full prompt it generates.

In [ ]:
# What demos did the optimizer select?
print(f"Number of demos: {len(optimized.demos)}")
print()
for i, demo in enumerate(optimized.demos):
    print(f"--- Demo {i+1} ---")
    print(f"  Q: {demo.get('question', 'N/A')}")
    print(f"  A: {demo.get('answer', 'N/A')}")
    print()

# Run one prediction to see the full few-shot prompt
_ = optimized(question=testset[0].question)
dspy.inspect_history(n=1)

## 11. Save and Load the Optimized Program

Optimized prompts are portable artifacts — save them as JSON and reload into a fresh predictor.

In [ ]:
import json as _json

# Save
optimized.save("optimized_city_qa.json")
print("Saved optimized program")

# Peek at the saved artifact
with open("optimized_city_qa.json") as f:
    saved = _json.load(f)
print(f"\nSaved artifact keys: {list(saved.keys()) if isinstance(saved, dict) else type(saved)}")
print(f"File size: {len(_json.dumps(saved))} bytes")

# Load into a fresh predictor
loaded = dspy.Predict(CityQA)
loaded.load("optimized_city_qa.json")

# Verify round-trip
loaded_score = evaluator(loaded)
print(f"\nOriginal optimized score: {optimized_score}")
print(f"Loaded program score:    {loaded_score}")

## 12. Try ChainOfThought

`ChainOfThought` wraps the signature with an extra `reasoning` field, forcing the model to think step-by-step before answering.

In [ ]:
cot = dspy.ChainOfThought(CityQA)

# Show reasoning on one example
result = cot(question=testset[0].question)
print(f"Question:  {testset[0].question}")
print(f"Reasoning: {result.reasoning}")
print(f"Answer:    {result.answer}")
print(f"Expected:  {testset[0].answer}")

# Evaluate
cot_score = evaluator(cot)
print(f"\nChainOfThought score: {cot_score}")
print(f"vs Baseline:          {baseline_score}")
print(f"vs Optimized Predict: {optimized_score}")

## 13. Build a Custom Module

Wrap the signature in a `dspy.Module` subclass. This is the standard pattern for composable DSPy programs — and it's what you'd optimize in a real pipeline.

In [ ]:
class CityExpert(dspy.Module):
    def __init__(self):
        self.answer = dspy.ChainOfThought(CityQA)

    def forward(self, question):
        return self.answer(question=question)

module = CityExpert()
module_score = evaluator(module)
print(f"Module (CoT) score: {module_score}")

# Optimize the module
module_optimizer = dspy.BootstrapFewShot(
    metric=city_metric,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)
optimized_module = module_optimizer.compile(module, trainset=trainset)
optimized_module_score = evaluator(optimized_module)

print(f"\nModule (CoT) score:           {module_score}")
print(f"Module (CoT) + optimized:     {optimized_module_score}")
print(f"Improvement:                  {optimized_module_score - module_score:+.2f}")

## 14. Improve the Metric with LLM-as-Judge

Our numeric metric is fast but brittle. Let's add an LLM-based faithfulness check: 70% numeric accuracy + 30% LLM judge.

In [ ]:
class FaithfulnessCheck(dspy.Signature):
    """Check if the answer is factually faithful and not hallucinated."""
    question: str = dspy.InputField()
    answer: str = dspy.InputField()
    expected_answer: str = dspy.InputField()
    is_faithful: bool = dspy.OutputField(desc="True if the answer is factually consistent with the expected answer")
    reason: str = dspy.OutputField(desc="Brief explanation of the judgment")

faithfulness_judge = dspy.Predict(FaithfulnessCheck)

def enhanced_metric(example, prediction, trace=None):
    """Numeric accuracy (70%) + LLM faithfulness judge (30%)."""
    expected = extract_number(example.answer)
    predicted = extract_number(prediction.answer)

    if expected is None or predicted is None or expected == 0:
        numeric_score = 0.0
    else:
        pct_error = abs(predicted - expected) / expected
        if pct_error <= 0.05:
            numeric_score = 1.0
        elif pct_error <= 0.10:
            numeric_score = 0.5
        else:
            numeric_score = 0.0

    # LLM judge component
    try:
        judgment = faithfulness_judge(
            question=example.question,
            answer=prediction.answer,
            expected_answer=example.answer
        )
        judge_score = 1.0 if judgment.is_faithful else 0.0
    except Exception:
        judge_score = 0.0

    score = 0.7 * numeric_score + 0.3 * judge_score

    if trace is not None:
        return score >= 0.5
    return score

# Re-evaluate the optimized module with the enhanced metric
enhanced_evaluator = dspy.Evaluate(
    devset=testset,
    metric=enhanced_metric,
    num_threads=2,
    display_progress=True,
    display_table=5,
)

enhanced_score = enhanced_evaluator(optimized_module)
print(f"\nOptimized module with numeric metric:   {optimized_module_score}")
print(f"Optimized module with enhanced metric:  {enhanced_score}")

## 15. Compare All Approaches

Summary table of every approach we tried.

In [ ]:
results = [
    ("Predict (baseline)", baseline_score),
    ("Predict + BootstrapFewShot", optimized_score),
    ("ChainOfThought", cot_score),
    ("Module (CoT)", module_score),
    ("Module (CoT) + BootstrapFewShot", optimized_module_score),
]

print(f"{'Approach':<40} {'Score':>8} {'vs Baseline':>12}")
print("─" * 62)
for name, score in results:
    delta = score - baseline_score
    bar = "█" * int(score / 2)
    print(f"{name:<40} {score:>8.2f} {delta:>+11.2f}  {bar}")

best_name, best_score = max(results, key=lambda x: x[1])
print(f"\n🏆 Best: {best_name} ({best_score:.2f})")

## Conclusion

In this module you:

1. **Declared** a city Q&A task as a typed DSPy signature
2. **Measured** quality with a percentage-error metric (5% / 10% thresholds)
3. **Optimized** prompts automatically with `BootstrapFewShot`
4. **Added reasoning** with `ChainOfThought`
5. **Composed** a custom module and optimized it end-to-end
6. **Enhanced** the metric with an LLM-as-judge faithfulness check

### Key Takeaway

> **Prompts are compiled artifacts, not hand-written strings.** Define what you want, measure quality, and let the optimizer find the best prompt.

### Next Steps

- Try `MIPROv2` for instruction optimization
- Build multi-stage pipelines (retrieve → reason → answer)
- Save optimized programs to S3 for deployment
- Add evaluation gates to CI/CD pipelines

### Resources

- [DSPy Documentation](https://dspy.ai/)
- [DSPy Optimizers Guide](https://dspy.ai/learn/optimization/optimizers/)
- [Amazon Bedrock Documentation](https://docs.aws.amazon.com/bedrock/)